# CodeReviewAgent - 智能代码审查助手

本项目演示如何使用HelloAgents框架构建一个智能代码审查助手。

## 📖 使用说明

- **快速体验**: 运行「第0部分」的快速演示
- **完整功能**: 依次运行第1-7部分,体验完整的代码审查流程

---

## 第0部分：快速演示 ⚡

如果你想快速了解项目功能,可以运行这个简化版本。

In [38]:
# 快速演示 - 导入库和配置
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
import ast
import os

# 配置LLM参数
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-72B-Instruct"
os.environ["LLM_API_KEY"] = "ms-9c264720-1dc2-4c5b-88b5-0244f39e8144"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

print("✅ 库导入和配置完成")

✅ 库导入和配置完成


In [39]:
# 快速演示 - 定义简单的代码分析工具
class QuickAnalysisTool(Tool):
    def __init__(self):
        super().__init__(
            name="quick_analysis",
            description="快速分析Python代码结构"
        )
    
    def run(self, parameters: Dict[str, Any]) -> str:
        code = parameters.get("code", "")
        if not code:
            return "错误：代码不能为空"
        
        try:
            tree = ast.parse(code)
            functions = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
            classes = [n.name for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]
            return f"发现{len(classes)}个类、{len(functions)}个函数: {', '.join(functions)}"
        except Exception as e:
            return f"代码解析失败: {str(e)}"
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="要分析的Python代码",
                required=True
            )
        ]

print("✅ 工具定义完成")

✅ 工具定义完成


In [40]:
# 快速演示 - 创建工具注册表和智能体
from hello_agents import ToolRegistry

# 创建工具注册表
quick_registry = ToolRegistry()
quick_registry.register_tool(QuickAnalysisTool())

# 创建智能体
quick_agent = SimpleAgent(
    name="快速审查助手",
    llm=HelloAgentsLLM(),
    system_prompt="你是代码审查助手,使用工具分析代码并给出简要建议。",
    tool_registry=quick_registry
)

print("✅ 智能体创建完成")
print(f"✅ 可用工具: {list(quick_registry._tools.keys())}")

✅ 工具 'quick_analysis' 已注册。
✅ 智能体创建完成
✅ 可用工具: ['quick_analysis']


In [41]:
# 快速演示 - 测试代码
test_code = """
def hello():
    print("Hello")

def world():
    print("World")

class Greeter:
    def greet(self):
        hello()
        world()
"""

print("=== 快速演示：分析测试代码 ===")
result = quick_agent.run(f"请分析这段代码:\n{test_code}")
print(result)
print("\n✅ 快速演示完成！")
print("\n💡 提示：继续运行下面的单元格,体验完整功能")

=== 快速演示：分析测试代码 ===
看来快速分析工具仍然无法正确解析你的代码，但根据你提供的代码内容，它看起来是正确的 Python 代码。这里有几个可能的原因和建议：

1. **编码问题**：确保文件是以 UTF-8 编码保存的。
2. **特殊字符**：检查代码中是否有不可见的特殊字符，例如在复制粘贴过程中可能带入的字符。
3. **行尾符**：确保行尾符是标准的换行符（LF），而不是回车换行符（CRLF）。

### 代码审查建议

1. **函数命名**：函数名 `hello` 和 `world` 遵循了 Python 的命名约定，使用小写字母和下划线。
2. **类命名**：类名 `Greeter` 使用了驼峰命名法（CamelCase），符合 PEP 8 规范。
3. **方法命名**：方法名 `greet` 也遵循了 Python 的命名约定。
4. **代码结构**：代码结构清晰，每个函数和类都有明确的职责。

### 改进建议

虽然代码本身没有明显的问题，但可以考虑以下几点：

1. **添加文档字符串**：为函数和类添加文档字符串，以便更好地描述它们的功能。
   ```python
   def hello():
       """Prints 'Hello'."""
       print("Hello")

   def world():
       """Prints 'World'."""
       print("World")

   class Greeter:
       """A class for greeting."""

       def greet(self):
           """Performs the greeting by calling hello() and world()."""
           hello()
           world()
   ```

2. **测试**：考虑为类和方法编写单元测试，以确保它们按预期工作。
   ```python
   import unittest

   class TestGreeter(unittest.TestCase):
       def test_greet(self):
           greet

---

# 完整版代码审查系统

下面是完整的代码审查系统,包含更强大的分析功能。

## 第1部分：环境配置

In [55]:
# 导入必要的库
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
#from trae_agent.tools.base import Tool, ToolParameter 
from typing import Dict, Any, List
import ast
import os

# 配置LLM参数
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-72B-Instruct"
os.environ["LLM_API_KEY"] = "ms-9c264720-1dc2-4c5b-88b5-0244f39e8144"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

print("✅ 环境配置完成")
print(f"✅ 使用模型: {os.getenv('LLM_MODEL_ID')}")
print(f"✅ API地址: {os.getenv('LLM_BASE_URL')}")

✅ 环境配置完成
✅ 使用模型: Qwen/Qwen2.5-72B-Instruct
✅ API地址: https://api-inference.modelscope.cn/v1/


## 第2部分：定义代码分析工具

In [56]:
class CodeAnalysisTool(Tool):
    """代码静态分析工具"""

    def __init__(self):
        super().__init__(
            name="code_analysis",
            description="分析Python代码的结构、复杂度和潜在问题"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        """分析代码并返回结果"""
        code = parameters.get("code", "")
        if not code:
            return "错误：代码不能为空"
        
        try:
            tree = ast.parse(code)

            # 统计信息
            functions = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
            classes = [node for node in ast.walk(tree) if isinstance(node, ast.ClassDef)]

            result = {
                "函数数量": len(functions),
                "类数量": len(classes),
                "代码行数": len(code.split('\n')),
                "函数列表": [f.name for f in functions],
                "类列表": [c.name for c in classes]
            }

            return str(result)
        except SyntaxError as e:
            return f"语法错误：{str(e)}"
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="要分析的Python代码",
                required=True
            )
        ]

print("✅ CodeAnalysisTool定义完成")

✅ CodeAnalysisTool定义完成


In [57]:
class StyleCheckTool(Tool):
    """代码风格检查工具"""

    def __init__(self):
        super().__init__(
            name="style_check",
            description="检查代码是否符合PEP 8规范"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        """检查代码风格"""
        code = parameters.get("code", "")
        if not code:
            return "错误：代码不能为空"
        
        issues = []

        lines = code.split('\n')
        for i, line in enumerate(lines, 1):
            # 检查行长度
            if len(line) > 79:
                issues.append(f"第{i}行：超过79个字符")

            # 检查缩进
            if line.startswith(' ') and not line.startswith('    '):
                if len(line) - len(line.lstrip()) not in [0, 4, 8, 12]:
                    issues.append(f"第{i}行：缩进不规范")

        if not issues:
            return "代码风格良好，符合PEP 8规范"
        return "发现以下问题：\n" + "\n".join(issues)
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="要检查的Python代码",
                required=True
            )
        ]

print("✅ StyleCheckTool定义完成")

✅ StyleCheckTool定义完成


In [58]:
class SecurityCheckTool(Tool):
    """Python 代码安全漏洞静态检查工具（基于简单词法+关键字）"""

    def __init__(self):
        super().__init__(
            name="security_check",
            description="扫描 Python 代码中潜在的高危函数与危险模式，提示用户注意安全风险"
        )

        # 6 类危险模式：①命令执行 ②文件写 ③反序列化 ④SQL ⑤动态导入 ⑥通配导入
        self.DANGER_MAP = {
            "os.system":              "命令执行，建议用 subprocess 并严格过滤外部输入",
            "os.popen":               "命令执行，建议用 subprocess 并严格过滤外部输入",
            "subprocess.call":        "命令执行，建议用 subprocess.run(..., shell=False) 并过滤输入",
            "subprocess.check_call":  "命令执行，建议用 subprocess.run(..., shell=False) 并过滤输入",
            "eval":                   "动态执行任意代码，极易产生 RCE，建议用 ast.literal_eval 或 JSON 解析",
            "exec":                   "动态执行任意代码，风险极高，尽量避免",
            "open":                   "文件写操作，请确认路径是否来自外部、是否做了校验/沙箱",
            "pickle.load":            "反序列化可导致任意代码执行，建议改用 json 或严格签名校验",
            "pickle.loads":           "反序列化可导致任意代码执行，建议改用 json 或严格签名校验",
            "marshal.load":           "反序列化可导致任意代码执行",
            "yaml.load":              "PyYAML 默认构造任意 Python 对象，请用 yaml.safe_load",
            "ctypes.CDLL":            "加载本地 so/dll，需确认路径可信",
            "__import__":             "动态导入，需确认模块名来源可信",
            "importlib.import_module": "动态导入，需确认模块名来源可信",
            "input":                  "Python2 中 input()=eval(raw_input())，Python3 虽安全但常被误用",
            "sqlite3.execute":        "SQL 注入入口，请使用参数化查询（? 占位符）",
            "cursor.execute":         "SQL 注入入口，请使用参数化查询",
            "from os import *":       "通配导入会污染命名空间，显式导入所需函数",
            "import os\n":            "（仅提醒）若结合外部输入拼接路径，请注意路径穿越",
        }

    # ----------- 核心扫描逻辑 -----------
    def run(self, parameters: Dict[str, Any]) -> str:
        code: str = parameters.get("code", "")
        if not code:
            return "错误：代码不能为空"

        issues: List[str] = []
        lines = code.splitlines()

        for lineno, line in enumerate(lines, 1):
            line_stripped = line.strip()
            if not line_stripped or line_stripped.startswith("#"):
                continue

            # 1) 子串匹配危险函数
            for pattern, msg in self.DANGER_MAP.items():
                if pattern in line:
                    issues.append(f"第 {lineno} 行：{line_stripped[:60]}...  →  {msg}")

            # 2) 简单正则补充：yaml.load( 前面没 safe_
            if "yaml.load(" in line and "safe_load" not in line:
                issues.append(f"第 {lineno} 行：检测到 yaml.load()，请改用 yaml.safe_load()")

            # 3) 通配导入
            if "from " in line and " import *" in line:
                issues.append(f"第 {lineno} 行：通配导入（import *）会污染命名空间，建议显式导入")

        if not issues:
            return "✅ 未检测到常见高危函数/模式，代码看起来相对安全（仍需结合业务场景人工复核）"
        return "⚠️ 发现潜在安全风险：\n" + "\n".join(issues)

    # ----------- 参数描述 -----------
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="待扫描的 Python 源代码",
                required=True
            )
        ]


print("✅ SecurityCheckTool 定义完成")

✅ SecurityCheckTool 定义完成


## 第3部分：创建智能体

In [59]:
# 导入工具注册表
from hello_agents import ToolRegistry

# 创建工具注册表
tool_registry = ToolRegistry()
tool_registry.register_tool(CodeAnalysisTool())
tool_registry.register_tool(StyleCheckTool())
tool_registry.register_tool(SecurityCheckTool())

# 初始化LLM
llm = HelloAgentsLLM()

# 定义系统提示词
system_prompt = """你是一位经验丰富的代码审查专家。你的任务是：

1. 使用code_analysis工具分析代码结构
2. 使用style_check工具检查代码风格
3. 基于分析结果，提供详细的审查报告

审查报告应包括：
- 代码结构分析
- 风格问题
- 潜在bug
- 性能优化建议
- 最佳实践建议

请以Markdown格式输出报告。"""

# 创建智能体
agent = SimpleAgent(
    name="代码审查助手",
    llm=llm,
    system_prompt=system_prompt,
    tool_registry=tool_registry
)

print("✅ 智能体创建完成")
print(f"智能体名称: {agent.name}")
print(f"可用工具: {list(tool_registry._tools.keys())}")

✅ 工具 'code_analysis' 已注册。
✅ 工具 'style_check' 已注册。
✅ 工具 'security_check' 已注册。
✅ 智能体创建完成
智能体名称: 代码审查助手
可用工具: ['code_analysis', 'style_check', 'security_check']


## 第4部分：读取示例代码

In [60]:
# 读取示例代码
with open("data/sample_code.py", "r", encoding="utf-8") as f:
# with open("data/orchestrator.py", "r", encoding="utf-8") as f:
    sample_code = f.read()

print("=== 待审查的代码 ===")
print(sample_code)
print("\n" + "="*50 + "\n")

=== 待审查的代码 ===
"""
示例代码：一个简单的用户管理系统
用于演示代码审查功能
"""

class UserManager:
    """用户管理类"""
    
    def __init__(self):
        self.users = []
    
    def add_user(self, name, age, email):
        """添加用户"""
        user = {"name": name, "age": age, "email": email}
        self.users.append(user)
        return True
    
    def get_user(self, name):
        """获取用户信息"""
        for user in self.users:
            if user["name"] == name:
                return user
        return None
    
    def delete_user(self, name):
        """删除用户"""
        for i, user in enumerate(self.users):
            if user["name"] == name:
                del self.users[i]
                return True
        return False

def calculate_average_age(users):
    """计算平均年龄"""
    total = 0
    for user in users:
        total += user["age"]
    return total / len(users)

def send_email(email, message):
    """发送邮件（模拟）"""
    print(f"发送邮件到 {email}: {message}")
    return True






## 第5部分：执行代码审查

In [62]:
# 执行代码审查
print("=== 开始代码审查 ===")
review_result = agent.run(f"请审查以下Python代码：\n\n```python\n{sample_code}\n```")

print(review_result)
print("代码审查完毕。。。")

=== 开始代码审查 ===
### 代码审查报告

#### 代码结构分析
在进行代码结构分析时，`code_analysis`工具报告了一个语法错误：“'[' was never closed”。这可能是因为在代码中存在未闭合的方括号，或者是一个解析错误。然而，从提供的代码来看，并没有明显的语法错误。这可能是由于工具对某些特定格式的处理不当造成的。为了确保准确性，我将再次手动检查代码结构。

#### 手动代码结构分析
- **UserManager 类**：
  - `__init__` 方法初始化了一个空列表 `self.users`，用于存储用户信息。
  - `add_user` 方法将新用户添加到 `self.users` 列表中。
  - `get_user` 方法通过用户名查找用户信息。
  - `delete_user` 方法通过用户名从 `self.users` 列表中删除用户。

- **calculate_average_age 函数**：
  - 计算给定用户列表的平均年龄。

- **send_email 函数**：
  - 模拟发送邮件的功能，打印一条消息。

#### 风格问题
根据 `style_check` 工具的结果，代码风格良好，符合 PEP 8 规范。没有发现明显的风格问题。

#### 潜在 bug
- **delete_user 方法**：
  - 在 `delete_user` 方法中，如果用户列表中有多个同名用户，删除操作只会删除第一个匹配的用户。这可能导致意外的行为。建议在删除用户前确认唯一性或提供更详细的错误信息。

- **calculate_average_age 函数**：
  - 如果 `users` 列表为空，`calculate_average_age` 函数会抛出 `ZeroDivisionError`。建议在计算前检查列表长度，避免除零错误。

#### 性能优化建议
- **用户查找和删除**：
  - 当用户数量较多时，线性查找（`get_user` 和 `delete_user` 中的 `for` 循环）效率较低。可以考虑使用字典（`dict`）来存储用户信息，以提高查找和删除的性能。例如，可以将 `self.users` 改为 `self.users = {}`，键为用户名，值为用户信息

## 第6部分：保存审查报告

In [23]:
# 保存审查报告
with open("outputs/review_report.md", "w", encoding="utf-8") as f:
    f.write(review_result)

print("\n✅ 审查报告已保存到 outputs/review_report.md")


✅ 审查报告已保存到 outputs/review_report.md


## 第7部分：总结与展望

### 实现的功能
- ✅ 代码结构分析
- ✅ PEP 8风格检查
- ✅ 智能审查报告生成

### 遇到的挑战
- 如何准确解析Python代码结构
- 如何设计合理的提示词让LLM生成高质量报告

### 未来改进方向
- 支持更多编程语言
- 添加安全漏洞检测
- 集成更多静态分析工具
- 支持批量文件审查